In [0]:
# Databricks notebook source
# ============================================
# TurbineTransformer notebook test harness
# ============================================
from pyspark.sql import Row
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    BooleanType,
)
from src.utils.turbine_transformer import TurbineTransformer


# ============================================
# Shared schemas
# ============================================

RAW_SCHEMA = StructType([
    StructField("timestamp", StringType(), True),
    StructField("turbine_id", StringType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True),
    StructField("power_output", DoubleType(), True),
    StructField("data_group", StringType(), True),
])

ANOMALY_SCHEMA = StructType([
    StructField("data_group", StringType(), True),
    StructField("turbine_id", StringType(), True),
    StructField("date", StringType(), True),
    StructField("power_output", DoubleType(), True),
    StructField("avg_power", DoubleType(), True),
    StructField("std_power", DoubleType(), True),
])

IMPUTED_SCHEMA = StructType([
    StructField("data_group", StringType(), True),
    StructField("turbine_id", StringType(), True),
    StructField("timestamp", StringType(), True),
    StructField("wind_speed", DoubleType(), True),
    StructField("wind_direction", DoubleType(), True),
    StructField("power_output", DoubleType(), True),
    StructField("date", StringType(), True),
    StructField("is_missing_row", BooleanType(), True),
])


# ============================================
# Helper functions
# ============================================

def make_df(rows, schema):
    """Create a DataFrame with an explicit schema."""
    return spark.createDataFrame(rows, schema=schema)


def assert_equal(actual, expected, message: str) -> None:
    """Simple assertion helper with readable output."""
    if actual != expected:
        raise AssertionError(f"{message} | expected={expected}, actual={actual}")


def print_test_result(test_name: str) -> None:
    """Print a consistent success message."""
    print(f"✅ {test_name} passed")


# ============================================
# Test 1: standardize_bronze
# ============================================

def test_standardize_bronze() -> None:
    rows = [
        ("2022-03-01 00:00:00", "1", 10.0, 180.0, 3.2, "1")
    ]
    df = make_df(rows, RAW_SCHEMA)

    result = TurbineTransformer.standardize_bronze(df)
    row = result.collect()[0]

    assert row["timestamp"] is not None, "timestamp should be cast to timestamp"
    assert str(result.schema["wind_speed"].dataType) == "DoubleType()", "wind_speed should be double"
    assert str(result.schema["power_output"].dataType) == "DoubleType()", "power_output should be double"
    assert row["date"] is not None, "date should be derived from timestamp"

    print_test_result("test_standardize_bronze")


# ============================================
# Test 2: deduplicate
# ============================================

def test_deduplicate() -> None:
    rows = [
        ("2022-03-01 00:00:00", "1", 10.0, 180.0, 3.2, "1"),
        ("2022-03-01 00:00:00", "1", 10.0, 180.0, 3.2, "1"),
    ]
    df = make_df(rows, RAW_SCHEMA)
    df = TurbineTransformer.standardize_bronze(df)

    result = TurbineTransformer.deduplicate(df)

    assert_equal(result.count(), 1, "deduplicate should remove duplicate turbine/timestamp rows")
    print_test_result("test_deduplicate")


# ============================================
# Test 3: split_valid_and_quarantine
# ============================================

def test_split_valid_and_quarantine() -> None:
    rows = [
        ("2022-03-01 00:00:00", "1", 10.0, 180.0, 3.2, "1"),   # valid
        (None, "2", 10.0, 180.0, 3.0, "1"),                    # missing timestamp
        ("2022-03-01 01:00:00", "3", -1.0, 180.0, 3.0, "1"),  # invalid wind speed
        ("2022-03-01 02:00:00", "4", 10.0, 180.0, 15.0, "1"), # invalid power
    ]
    df = make_df(rows, RAW_SCHEMA)
    df = TurbineTransformer.standardize_bronze(df)

    valid_df, quarantine_df = TurbineTransformer.split_valid_and_quarantine(df)

    assert_equal(valid_df.count(), 1, "only one row should be valid")
    assert_equal(quarantine_df.count(), 3, "three rows should be quarantined")

    reasons = {r["quarantine_reason"] for r in quarantine_df.select("quarantine_reason").collect()}
    expected_reasons = {
        "missing_timestamp",
        "negative_wind_speed",
        "power_output_out_of_range",
    }
    assert_equal(reasons, expected_reasons, "quarantine reasons should match expected values")

    print_test_result("test_split_valid_and_quarantine")


# ============================================
# Test 4: calculate_daily_summary
# ============================================

def test_calculate_daily_summary() -> None:
    rows = [
        ("2022-03-01 00:00:00", "1", 10.0, 180.0, 3.0, "1"),
        ("2022-03-01 01:00:00", "1", 11.0, 181.0, 5.0, "1"),
        ("2022-03-01 02:00:00", "1", 12.0, 182.0, 4.0, "1"),
    ]
    df = make_df(rows, RAW_SCHEMA)
    df = TurbineTransformer.standardize_bronze(df)

    summary_df = TurbineTransformer.calculate_daily_summary(df)
    row = summary_df.collect()[0]

    assert_equal(row["min_power"], 3.0, "min_power should be 3.0")
    assert_equal(row["max_power"], 5.0, "max_power should be 5.0")
    assert round(row["avg_power"], 2) == 4.0, "avg_power should be 4.0"
    assert_equal(row["reading_count"], 3, "reading_count should be 3")

    print_test_result("test_calculate_daily_summary")


# ============================================
# Test 5: flag_anomalies
# ============================================

def test_flag_anomalies() -> None:
    rows = [
        ("1", "1", "2022-03-01", 5.0, 5.0, 0.5),  # not anomaly
        ("1", "1", "2022-03-01", 7.0, 5.0, 0.5),  # anomaly
    ]
    df = make_df(rows, ANOMALY_SCHEMA).withColumn("date", F.to_date("date"))

    flagged_df = TurbineTransformer.flag_anomalies(df)

    anomaly_count = flagged_df.filter(F.col("is_anomaly") == True).count()
    assert_equal(anomaly_count, 1, "exactly one row should be flagged as anomaly")

    print_test_result("test_flag_anomalies")


# ============================================
# Test 6: calculate_missing_intervals
# ============================================

def test_calculate_missing_intervals() -> None:
    rows = [
        ("2022-03-01 00:00:00", "1", 10.0, 180.0, 3.0, "1"),
        ("2022-03-01 01:00:00", "1", 11.0, 181.0, 3.4, "1"),
        ("2022-03-01 03:00:00", "1", 12.0, 182.0, 3.8, "1"),
    ]
    df = make_df(rows, RAW_SCHEMA)
    df = TurbineTransformer.standardize_bronze(df)

    missing_df = TurbineTransformer.calculate_missing_intervals(df, frequency="1 hour")

    assert_equal(missing_df.count(), 1, "one interval should be missing")

    missing_ts = missing_df.collect()[0]["missing_timestamp"]
    assert str(missing_ts)[:19] == "2022-03-01 02:00:00", "missing timestamp should be 02:00:00"

    print_test_result("test_calculate_missing_intervals")


# ============================================
# Test 7: forward_fill_short_gaps
# ============================================

def test_forward_fill_short_gaps() -> None:
    rows = [
        ("1", "1", "2022-03-01 00:00:00", 10.0, 180.0, 3.0, "2022-03-01", False),
        ("1", "1", "2022-03-01 01:00:00", None, None, None, "2022-03-01", True),
        ("1", "1", "2022-03-01 02:00:00", None, None, None, "2022-03-01", True),
        ("1", "1", "2022-03-01 03:00:00", 11.0, 181.0, 3.5, "2022-03-01", False),
    ]
    df = make_df(rows, IMPUTED_SCHEMA)
    df = (
        df.withColumn("timestamp", F.to_timestamp("timestamp"))
          .withColumn("date", F.to_date("date"))
    )

    result = TurbineTransformer.forward_fill_short_gaps(df, max_gap_hours=3)

    filled_rows = (
        result.filter(F.col("is_imputed") == True)
              .orderBy("timestamp")
              .collect()
    )

    assert_equal(len(filled_rows), 2, "two rows should be imputed")
    assert_equal(filled_rows[0]["power_output"], 3.0, "first gap should forward-fill power_output")
    assert_equal(filled_rows[1]["power_output"], 3.0, "second gap should forward-fill power_output")

    print_test_result("test_forward_fill_short_gaps")


# ============================================
# Run all tests
# ============================================

test_standardize_bronze()
test_deduplicate()
test_split_valid_and_quarantine()
test_calculate_daily_summary()
test_flag_anomalies()
test_calculate_missing_intervals()
test_forward_fill_short_gaps()

print("🎉 All TurbineTransformer notebook tests passed")